# Imports

In [11]:
import warnings
# warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", module="tqdm")

import os
from tqdm.auto import tqdm

from os.path import join
from PIL import Image
import matlab.engine
import numpy as np
from pathlib import Path
from utils.ImageResizer import ImageResizer


# Настройка движка матлаб

In [12]:
# Start engine
eng = matlab.engine.start_matlab()

# S_UNIWARD
path_S_UNIWARD = r'S-UNIWARD_matlab\matlab_MEX'
eng.addpath(path_S_UNIWARD, nargout=0)

# WOW
path_WOW = r'WOW_matlab\matlab_MEX'
eng.addpath(path_WOW, nargout=0)

# HUGO 
path_HUGO = r'HUGO_cpp\matlab\matlab_MEX'
eng.addpath(path_HUGO, nargout=0)

# MiPOD
path_HUGO = r'MiPOD_matlab'
eng.addpath(path_HUGO, nargout=0)


Exception ignored in: <function MatlabEngine.__del__ at 0x000002009CD06700>
Traceback (most recent call last):
  File "c:\Users\Администратор\OneDrive\Рабочий стол\diplom\venv\Lib\site-packages\matlab\engine\matlabengine.py", line 250, in __del__
    self.exit()
  File "c:\Users\Администратор\OneDrive\Рабочий стол\diplom\venv\Lib\site-packages\matlab\engine\matlabengine.py", line 232, in exit
    pythonengine.closeMATLAB(self.__dict__["_matlab"])
SystemError: MATLAB process cannot be terminated.


# Utilities

In [4]:
import matplotlib.pyplot as plt

def plot_pgm_image(pillow_image, title='PGM Image'):
    plt.figure(figsize=(8, 6))
    plt.imshow(pillow_image, cmap='gray', vmin=0, vmax=255)
    plt.colorbar(label='Intensity')
    plt.title(title)
    
    plt.axis('off')  # Скрываем оси
    plt.show()

def plot_pgm_diff(pillow_stego_image, pillow_cover_image, title='embedding changes: +1 = white, -1 = black'):
    cover = np.array(pillow_cover_image.convert('L'))
    stego = np.array(pillow_stego_image.convert('L'))
        
    diff = np.abs(stego.astype(np.float64) - cover.astype(np.float64))
        
    plt.figure(figsize=(8, 6))
    plt.imshow(diff, cmap='hot', interpolation='nearest')
    plt.colorbar(label='Magnitude of change')
    plt.title(title)

    plt.axis('off')     
    plt.show()

# Image datasets

## BOSSbase

In [13]:
image_collection_folder_path = r'..\..\images\BOSSbase'
image_collection_folder_path = Path(image_collection_folder_path)

cover_raw_image_folder_path = image_collection_folder_path / 'cover_raw'
cover_image_folder_path = image_collection_folder_path / 'cover'
stego_image_folder_path = image_collection_folder_path / 'stego'
stego_image_folder_path.mkdir(exist_ok=True)

os.path.exists(image_collection_folder_path)

True

# Тестируем все алгоритмы встраивания

## S-UNIWARD (<1 sec per image)

In [12]:
filename = '1.pgm'
algorithm_name = 's-uniward'
bpp = 0.2

stego_image_filename = f'{algorithm_name}_{bpp}bpp_{filename}'
stego_image_path = stego_image_folder_path / stego_image_filename

cover_image_path = cover_image_folder_path / filename
cover_pil = Image.open(cover_image_path).convert('L')
cover_np = np.array(cover_pil)

In [13]:
# MATLAB API call
cover_matlab = matlab.uint8(cover_np.tolist())
payload =      matlab.single(bpp)  

stego_matlab, distortion = eng.S_UNIWARD(cover_matlab, payload, nargout=2)

In [82]:
stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
stego_image = Image.fromarray(stego_np.astype(np.uint8))
stego_image.save(stego_image_path)

In [84]:
# stego_image = Image.open(stego_image_path)
# plot_pgm_image(stego_image, title='Stego image')

# cover_image = Image.open(cover_image_path)
# plot_pgm_image(cover_image, title='Cover image')

# plot_pgm_diff(stego_image, cover_image)

## WOW (<1 sec per image)

In [ ]:
filename = '1.pgm'
algorithm_name = 'wow'
bpp = 0.2

stego_image_filename = f'{algorithm_name}_{bpp}bpp_{filename}'
stego_image_path = stego_image_folder_path / stego_image_filename


cover_image_path = cover_image_folder_path / filename
cover_pil = Image.open(cover_image_path).convert('L')
cover_np = np.array(cover_pil)

In [ ]:
# MATLAB API call
cover_matlab = matlab.uint8(cover_np.tolist())
payload =      matlab.single(bpp)  

stego_matlab, distortion = eng.WOW(cover_matlab, payload, nargout=2)

In [ ]:
stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
stego_image = Image.fromarray(stego_np.astype(np.uint8))
stego_image.save(stego_image_path)

In [56]:
# stego_image = Image.open(stego_image_path)
# plot_pgm_image(stego_image, title='Stego image')

# cover_image = Image.open(cover_image_path)
# plot_pgm_image(cover_image, title='Cover image')

# plot_pgm_diff(stego_image, cover_image)

## HUGO (<1 sec per image)

In [85]:
filename = '1.pgm'
algorithm_name = 'hugo'
bpp = 0.2

stego_image_filename = f'{algorithm_name}_{bpp}bpp_{filename}'
stego_image_path = stego_image_folder_path / stego_image_filename


cover_image_path = cover_image_folder_path / filename
cover_pil = Image.open(cover_image_path).convert('L')
cover_np = np.array(cover_pil)

In [86]:
# MATLAB API call
cover_matlab = matlab.uint8(cover_np.tolist())
payload =      matlab.single(bpp)  

stego_matlab, distortion = eng.HUGO_like(cover_matlab, payload, nargout=2)

In [87]:
stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
stego_image = Image.fromarray(stego_np.astype(np.uint8))
stego_image.save(stego_image_path)

In [76]:
# stego_image = Image.open(stego_image_path)
# plot_pgm_image(stego_image, title='Stego image')

# cover_image = Image.open(cover_image_path)
# plot_pgm_image(cover_image, title='Cover image')

# plot_pgm_diff(stego_image, cover_image)

## <div style="color: #ff0000;">MiPOD (6 sec per image)</div>

In [ ]:
filename = '1.pgm'
algorithm_name = 'mipod'
bpp = 0.2

stego_image_filename = f'{algorithm_name}_{bpp}bpp_{filename}'
stego_image_path = stego_image_folder_path / stego_image_filename


cover_image_path = cover_image_folder_path / filename
cover_pil = Image.open(cover_image_path).convert('L')
cover_np = np.array(cover_pil)

In [ ]:
# MATLAB API call
cover_matlab = matlab.uint8(cover_np.tolist())
payload =      matlab.single(bpp)  

stego_matlab, p_change, change_rate = eng.MiPOD(cover_matlab, payload, nargout=3)

In [ ]:
stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
stego_image = Image.fromarray(stego_np.astype(np.uint8))
stego_image.save(stego_image_path)

In [77]:
# stego_image = Image.open(stego_image_path)
# plot_pgm_image(stego_image, title='Stego image')

# cover_image = Image.open(cover_image_path)
# plot_pgm_image(cover_image, title='Cover image')

# plot_pgm_diff(stego_image, cover_image)

# Array embedding

In [14]:
fn_list = os.listdir(cover_raw_image_folder_path)
fn_list = fn_list[:7000]
path_list = [cover_raw_image_folder_path / fn for fn in fn_list]

param_grid = {
    "path": path_list,
    "bpp": [0.4],
    "algorithm_name": ['s-uniward'],
    "resize_strategy": ['center_crop'],
    "target_size": [(256, 256)],
}

from itertools import product

keys = param_grid.keys()
values = param_grid.values()

param_dict_list = [dict(zip(keys, v)) for v in product(*values)]

print('Всего комбинаций:', len(param_dict_list))

Всего комбинаций: 7000


In [ ]:
for param_dict in tqdm(param_dict_list):
    path = param_dict['path']
    algorithm_name = param_dict['algorithm_name']
    bpp = param_dict['bpp']
    resize_strategy = param_dict['resize_strategy']
    target_size = param_dict['target_size']
    W, H = target_size

    filename = path.name

    cover_raw_image_path = cover_raw_image_folder_path / filename

    cover_image_filename = f'{resize_strategy}_{W}_{H}_{filename}'
    cover_image_path = cover_image_folder_path / cover_image_filename
    
    stego_image_filename = f'{resize_strategy}_{W}_{H}_{algorithm_name}_{bpp}bpp_{filename}'
    stego_image_path = stego_image_folder_path / stego_image_filename
    
    if os.path.exists(stego_image_path) and os.path.exists(cover_image_path):
        continue


    cover_pil = Image.open(cover_raw_image_path).convert('L')
    cover_pil_resized = ImageResizer(target_size=target_size, strategy=resize_strategy)(cover_pil)
    cover_np = np.array(cover_pil_resized)

    # MATLAB API call
    cover_matlab = matlab.uint8(cover_np.tolist())
    payload =      matlab.single(bpp)  

    if algorithm_name == 's-uniward':
        stego_matlab, distortion = eng.S_UNIWARD(cover_matlab, payload, nargout=2)
    elif algorithm_name == 'wow':
        stego_matlab, distortion = eng.WOW(cover_matlab, payload, nargout=2)
    elif algorithm_name == 'hugo':
        stego_matlab, distortion = eng.HUGO_like(cover_matlab, payload, nargout=2)
    else:
        raise NotImplementedError(f'Для {algorithm_name} нет подходящего алгоритма встраивания!')

    # Save image [stego]
    stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
    stego_image = Image.fromarray(stego_np.astype(np.uint8))
    stego_image.save(stego_image_path)

    # Save image [cover]
    cover_pil_resized.save(cover_image_path)

100%|██████████| 7000/7000 [19:40<00:00,  5.93it/s]  


## YeNet old embedding dirs prepare

In [15]:
len(path_list)

train_samples_num = 1024
valid_samples_num = 128

train_cover_path = Path(r'..\..\images\BOSSbase\YeNet\train_cover')
train_stego_path = Path(r'..\..\images\BOSSbase\YeNet\train_stego')
valid_cover_path = Path(r'..\..\images\BOSSbase\YeNet\valid_cover')
valid_stego_path = Path(r'..\..\images\BOSSbase\YeNet\valid_stego')

bpp = 0.4
target_size = (256, 256)
resize_strategy = 'center_crop'

for path in tqdm(path_list[0:train_samples_num]):
    cover_pil = Image.open(path).convert('L')
    cover_pil_resized = ImageResizer(target_size=target_size, strategy=resize_strategy)(cover_pil)
    cover_np = np.array(cover_pil_resized)

    # MATLAB API call
    cover_matlab = matlab.uint8(cover_np.tolist())
    payload =      matlab.single(bpp)  
    stego_matlab, distortion = eng.S_UNIWARD(cover_matlab, payload, nargout=2)

    stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
    stego_image = Image.fromarray(stego_np.astype(np.uint8))

    filename = path.name
    train_cover_image_path = train_cover_path / filename
    train_stego_image_path = train_stego_path / filename

    # Save Image
    cover_pil_resized.save(train_cover_image_path)
    stego_image.save(train_stego_image_path)
    

for path in tqdm(path_list[train_samples_num:train_samples_num+valid_samples_num]):
    cover_pil = Image.open(path).convert('L')
    cover_pil_resized = ImageResizer(target_size=target_size, strategy=resize_strategy)(cover_pil)
    cover_np = np.array(cover_pil_resized)

    # MATLAB API call
    cover_matlab = matlab.uint8(cover_np.tolist())
    payload =      matlab.single(bpp)  
    stego_matlab, distortion = eng.S_UNIWARD(cover_matlab, payload, nargout=2)

    stego_np = np.array(stego_matlab._data).reshape(stego_matlab.size[::1]).T
    stego_image = Image.fromarray(stego_np.astype(np.uint8))

    filename = path.name
    valid_cover_image_path = valid_cover_path / filename
    valid_stego_image_path = valid_stego_path / filename

    # Save Image
    cover_pil_resized.save(valid_cover_image_path)
    stego_image.save(valid_stego_image_path)
    

100%|██████████| 128/128 [00:32<00:00,  3.92it/s]


In [16]:
eng.exit()